In [3]:
import cv2
import numpy as np
import tensorflow as tf
import os

In [5]:
import cv2
import numpy as np
import os

ini_path = "train_dataset"

# Create the processed directories if they don't exist
train_processed_dir = "train_processed"
test_processed_dir = "test_processed"

# Ensure the directories exist
if not os.path.exists(train_processed_dir):
    os.makedirs(train_processed_dir)

if not os.path.exists(test_processed_dir):
    os.makedirs(test_processed_dir)

for (dirpath, dirnames, filenames) in os.walk(ini_path):
    for dirname in dirnames:
        # Create subdirectories for each class in train and test processed directories
        train_class_dir = os.path.join(train_processed_dir, dirname)
        test_class_dir = os.path.join(test_processed_dir, dirname)
        
        if not os.path.exists(train_class_dir):
            os.makedirs(train_class_dir)
        
        if not os.path.exists(test_class_dir):
            os.makedirs(test_class_dir)
        
        for (direcpath, direcnames, files) in os.walk(os.path.join(ini_path, dirname)):
            i = 0
            for file in files:
                actual_path = os.path.join(ini_path, dirname, file)

                frame = cv2.imread(actual_path)
                frame = cv2.resize(frame, (64, 64))

                converted = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)  # Convert from RGB to HSV

                lowerBoundary = np.array([0, 40, 30], dtype="uint8")
                upperBoundary = np.array([43, 255, 254], dtype="uint8")

                skinMask = cv2.inRange(converted, lowerBoundary, upperBoundary)
                skinMask = cv2.medianBlur(skinMask, 5)
                skin = cv2.bitwise_and(frame, frame, mask=skinMask)

                bw_image = cv2.cvtColor(skin, cv2.COLOR_HSV2BGR)
                bw_image = cv2.cvtColor(skin, cv2.COLOR_BGR2GRAY)
                bw_image = cv2.GaussianBlur(bw_image, (5, 5), 0)

                if i % 5 == 0:
                    directory = test_class_dir
                else:
                    directory = train_class_dir

                curr_path = os.path.join(directory, file)
                cv2.imwrite(curr_path, bw_image)
                i += 1
